# Bootstrap Confidence Intervals for CICIoMT2024 IoMT Paper

هذا الدفتر مخصص فقط لحساب **Bootstrap 95% Confidence Intervals** من مخرجات التشغيل النهائي الموجودة في Google Drive.

لا يعيد تدريب OCSVM أو LOF أو Isolation Forest.  
لا يعيد قراءة الداتا الخام.  
يعتمد على ملفات النتائج النهائية مثل:

- `metrics/metrics_main.csv`
- `publication_checks/attack_only_metrics_excluding_benign.csv`
- `predictions/test_predictions.parquet` أو `predictions/test_predictions.csv`

المخرجات تحفظ في:

- `publication_checks/bootstrap_ci/`
- ملف ZIP مستقل باسم `bootstrap_ci_results_*.zip`


In [1]:
# وظيفة الخلية: ربط Google Drive حتى يقرأ الدفتر مخرجات التشغيل النهائي ويحفظ نتائج Bootstrap في نفس بيئة Drive.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# وظيفة الخلية: استيراد المكتبات وضبط إعدادات Bootstrap العامة بدون أي إعادة تدريب للنماذج.

import os
import re
import json
import math
import time
import zipfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score

# المسار العام لـ Google Drive في Colab.
DRIVE_ROOT = "/content/drive/MyDrive"

# ضع هنا مسار مجلد النتائج إذا أردت تحديده يدوياً.
# يجب أن يكون هذا المجلد يحتوي على metrics/ و publication_checks/ و predictions/.
# مثال:
# MANUAL_RESULTS_ROOT = "/content/drive/MyDrive/IoMT/publication_full_20260506_013718"
MANUAL_RESULTS_ROOT = ""

# ضع هنا مسار ZIP النتائج إذا كانت النتائج محفوظة كأرشيف مضغوط.
# مثال:
# MANUAL_RESULTS_ZIP = "/content/drive/MyDrive/publication_full_20260506_013718.zip"
MANUAL_RESULTS_ZIP = ""

# عدد مرات Bootstrap للمقاييس المبنية على confusion matrix.
# هذا سريع جداً لأنه يستخدم counts فقط.
COUNT_BOOTSTRAP_B = 2000

# تفعيل Bootstrap للـ ROC-AUC و PR-AUC من anomaly_score.
# هذا أبطأ لأنه يقرأ predictions ويعيد حساب AUC/AP على عينات Bootstrap.
RUN_SCORE_BASED_AUC_CI = True

# عدد مرات Bootstrap للـ ROC-AUC و PR-AUC.
# إذا احتجت نتيجة أسرع خفّضها إلى 100. إذا أردت أقوى للمراجعة ارفعها إلى 500.
SCORE_BOOTSTRAP_B = 200

# النموذج/الإعداد الرئيسي للـ score-based CI.
TARGET_SETTING = "dedup_on"
TARGET_MODEL = "ocsvm"

# Bootstrap seed لتكرار النتائج.
RANDOM_SEED = 42

# إذا كان CSV ضخماً، يستخدم الدفتر القراءة على دفعات.
CSV_CHUNKSIZE = 1_000_000

# إنشاء مجلد عمل محلي سريع.
LOCAL_WORKDIR = "/content/bootstrap_ci_work"
os.makedirs(LOCAL_WORKDIR, exist_ok=True)

print("COUNT_BOOTSTRAP_B =", COUNT_BOOTSTRAP_B)
print("RUN_SCORE_BASED_AUC_CI =", RUN_SCORE_BASED_AUC_CI)
print("SCORE_BOOTSTRAP_B =", SCORE_BOOTSTRAP_B)
print("TARGET =", TARGET_SETTING, TARGET_MODEL)


COUNT_BOOTSTRAP_B = 2000
RUN_SCORE_BASED_AUC_CI = True
SCORE_BOOTSTRAP_B = 200
TARGET = dedup_on ocsvm


In [3]:
# وظيفة الخلية: تعريف أدوات البحث عن مجلد النتائج أو ملف ZIP داخل Google Drive بشكل تلقائي.

def depth_from_root(path, root):
    path = os.path.abspath(path)
    root = os.path.abspath(root)
    rel = os.path.relpath(path, root)
    if rel == ".":
        return 0
    return rel.count(os.sep) + 1

def is_results_root(path):
    path = Path(path)
    return (
        (path / "metrics" / "metrics_main.csv").exists()
        and (path / "publication_checks").exists()
        and (path / "predictions").exists()
    )

def zip_contains_results(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            names = set(z.namelist())
            return (
                "metrics/metrics_main.csv" in names
                and "publication_checks/attack_only_metrics_excluding_benign.csv" in names
                and (
                    "predictions/test_predictions.parquet" in names
                    or "predictions/test_predictions.csv" in names
                )
            )
    except Exception:
        return False

def find_result_roots(base_dir, max_depth=7, max_hits=20):
    hits = []
    skip_names = {
        ".Trash",
        ".shortcut-targets-by-id",
        ".file-revisions-by-id",
        "__MACOSX",
        ".ipynb_checkpoints",
    }
    for root, dirs, files in os.walk(base_dir):
        dirs[:] = [d for d in dirs if d not in skip_names]
        if depth_from_root(root, base_dir) > max_depth:
            dirs[:] = []
            continue
        if is_results_root(root):
            hits.append(root)
            if len(hits) >= max_hits:
                break
    return hits

def find_result_zips(base_dir, max_depth=7, max_hits=20):
    hits = []
    skip_names = {
        ".Trash",
        ".shortcut-targets-by-id",
        ".file-revisions-by-id",
        "__MACOSX",
        ".ipynb_checkpoints",
    }
    candidate_name_re = re.compile(r"(publication|full|iomt|bootstrap|results)", re.IGNORECASE)
    for root, dirs, files in os.walk(base_dir):
        dirs[:] = [d for d in dirs if d not in skip_names]
        if depth_from_root(root, base_dir) > max_depth:
            dirs[:] = []
            continue
        for f in files:
            if not f.lower().endswith(".zip"):
                continue
            if not candidate_name_re.search(f):
                continue
            fp = os.path.join(root, f)
            if zip_contains_results(fp):
                hits.append(fp)
                if len(hits) >= max_hits:
                    return hits
    return hits

print("Search helpers ready.")


Search helpers ready.


In [4]:
# وظيفة الخلية: تحديد مصدر النتائج النهائي تلقائياً أو عبر المسار اليدوي.

def resolve_results_source():
    if MANUAL_RESULTS_ROOT:
        if not is_results_root(MANUAL_RESULTS_ROOT):
            raise FileNotFoundError(f"MANUAL_RESULTS_ROOT does not look like a result root: {MANUAL_RESULTS_ROOT}")
        return {"mode": "folder", "path": MANUAL_RESULTS_ROOT, "drive_output_parent": MANUAL_RESULTS_ROOT}

    if MANUAL_RESULTS_ZIP:
        if not os.path.exists(MANUAL_RESULTS_ZIP):
            raise FileNotFoundError(f"MANUAL_RESULTS_ZIP not found: {MANUAL_RESULTS_ZIP}")
        if not zip_contains_results(MANUAL_RESULTS_ZIP):
            raise FileNotFoundError(f"ZIP does not contain expected result files: {MANUAL_RESULTS_ZIP}")
        return {"mode": "zip", "path": MANUAL_RESULTS_ZIP, "drive_output_parent": os.path.dirname(MANUAL_RESULTS_ZIP)}

    print("Searching Google Drive for extracted result folders...")
    roots = find_result_roots(DRIVE_ROOT)
    if roots:
        print("Found result folder candidates:")
        for i, p in enumerate(roots, 1):
            print(f"{i}. {p}")
        return {"mode": "folder", "path": roots[0], "drive_output_parent": roots[0]}

    print("No extracted result folder found. Searching Google Drive for result ZIP files...")
    zips = find_result_zips(DRIVE_ROOT)
    if zips:
        print("Found result ZIP candidates:")
        for i, p in enumerate(zips, 1):
            print(f"{i}. {p}")
        return {"mode": "zip", "path": zips[0], "drive_output_parent": os.path.dirname(zips[0])}

    raise FileNotFoundError(
        "Could not find result folder or ZIP automatically. "
        "Set MANUAL_RESULTS_ROOT or MANUAL_RESULTS_ZIP in the configuration cell."
    )

SOURCE_INFO = resolve_results_source()
print("\\nSelected source:")
print(json.dumps(SOURCE_INFO, indent=2, ensure_ascii=False))


Searching Google Drive for extracted result folders...
No extracted result folder found. Searching Google Drive for result ZIP files...
Found result ZIP candidates:
1. /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/publication_full_20260506_013718.zip
\nSelected source:
{
  "mode": "zip",
  "path": "/content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/publication_full_20260506_013718.zip",
  "drive_output_parent": "/content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full"
}


In [5]:
# وظيفة الخلية: تجهيز ملفات النتائج محلياً للقراءة السريعة إذا كان المصدر ZIP، أو استخدام المجلد مباشرة إذا كان مفكوكاً.

if SOURCE_INFO["mode"] == "folder":
    RESULTS_ROOT = SOURCE_INFO["path"]
    OUTPUT_PARENT = os.path.join(RESULTS_ROOT, "publication_checks", "bootstrap_ci")
    os.makedirs(OUTPUT_PARENT, exist_ok=True)

else:
    extract_dir = os.path.join(LOCAL_WORKDIR, "extracted_results")
    if os.path.exists(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)

    required_prefixes = [
        "metrics/",
        "publication_checks/",
        "predictions/test_predictions.parquet",
        "predictions/test_predictions.csv",
    ]

    with zipfile.ZipFile(SOURCE_INFO["path"], "r") as z:
        names = z.namelist()
        to_extract = []
        for n in names:
            if n.startswith("metrics/") or n.startswith("publication_checks/"):
                to_extract.append(n)
            elif n in ("predictions/test_predictions.parquet", "predictions/test_predictions.csv"):
                to_extract.append(n)
        print("Extracting", len(to_extract), "files from ZIP...")
        for n in to_extract:
            z.extract(n, extract_dir)

    RESULTS_ROOT = extract_dir
    OUTPUT_PARENT = os.path.join(SOURCE_INFO["drive_output_parent"], "bootstrap_ci_results_from_zip")
    os.makedirs(OUTPUT_PARENT, exist_ok=True)

print("RESULTS_ROOT =", RESULTS_ROOT)
print("OUTPUT_PARENT =", OUTPUT_PARENT)


Extracting 21 files from ZIP...
RESULTS_ROOT = /content/bootstrap_ci_work/extracted_results
OUTPUT_PARENT = /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip


In [6]:
# وظيفة الخلية: تحميل جداول المقاييس الأساسية التي سنحسب منها فواصل الثقة للمقاييس threshold-based.

metrics_main_path = os.path.join(RESULTS_ROOT, "metrics", "metrics_main.csv")
attack_only_path = os.path.join(RESULTS_ROOT, "publication_checks", "attack_only_metrics_excluding_benign.csv")
benign_sensitivity_path = os.path.join(RESULTS_ROOT, "metrics", "benign_sensitivity_metrics.csv")

metrics_main = pd.read_csv(metrics_main_path)
attack_only_metrics = pd.read_csv(attack_only_path) if os.path.exists(attack_only_path) else pd.DataFrame()
benign_sensitivity_metrics = pd.read_csv(benign_sensitivity_path) if os.path.exists(benign_sensitivity_path) else pd.DataFrame()

print("metrics_main:", metrics_main.shape)
print(metrics_main[["setting", "model", "tn", "fp", "fn", "tp", "f1", "mcc", "fpr", "balanced_accuracy"]])

print("\\nattack_only_metrics:", attack_only_metrics.shape)
if not attack_only_metrics.empty:
    print(attack_only_metrics[["setting", "model", "tn", "fp", "fn", "tp", "f1", "mcc", "fpr", "balanced_accuracy"]])


metrics_main: (6, 20)
     setting             model     tn    fp      fn       tp        f1  \
0  dedup_off             ocsvm  73731  3900       0  1614182  0.998793   
1  dedup_off               lof  73897  3734   12244  1601938  0.995038   
2  dedup_off  isolation_forest  73736  3895  805646   808536  0.666391   
3   dedup_on             ocsvm  68601  3592       0  1612111  0.998887   
4   dedup_on               lof  68570  3623   10580  1601531  0.995585   
5   dedup_on  isolation_forest  68607  3586  794852   817259  0.671824   

        mcc       fpr  balanced_accuracy  
0  0.973382  0.050238           0.974881  
1  0.898830  0.048099           0.972158  
2  0.188765  0.050173           0.725361  
3  0.973721  0.049756           0.975122  
4  0.902795  0.050185           0.971626  
5  0.185299  0.049672           0.728639  
\nattack_only_metrics: (6, 19)
     setting             model     tn    fp      fn       tp        f1  \
0   dedup_on  isolation_forest  68607  3586  763032  

In [7]:
# وظيفة الخلية: تعريف دوال حساب المقاييس من مصفوفة الالتباس وحساب Bootstrap CI اعتماداً على counts فقط.

def safe_div(num, den):
    return float(num) / float(den) if den else np.nan

def metrics_from_counts(tn, fp, fn, tp):
    tn, fp, fn, tp = map(float, [tn, fp, fn, tp])
    n = tn + fp + fn + tp

    accuracy = safe_div(tp + tn, n)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    fpr = safe_div(fp, fp + tn)
    f1 = safe_div(2 * precision * recall, precision + recall) if not (np.isnan(precision) or np.isnan(recall)) else np.nan
    balanced_accuracy = np.nanmean([recall, specificity])

    denom = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    mcc = ((tp * tn) - (fp * fn)) / math.sqrt(denom) if denom > 0 else np.nan

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "fpr": fpr,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": balanced_accuracy,
    }

def bootstrap_ci_from_confusion_counts(tn, fp, fn, tp, B=2000, seed=42):
    rng = np.random.default_rng(seed)
    counts = np.array([tn, fp, fn, tp], dtype=np.int64)
    n = int(counts.sum())
    if n <= 0:
        raise ValueError("Empty confusion matrix")

    probs = counts / counts.sum()
    boot_counts = rng.multinomial(n=n, pvals=probs, size=B)

    records = []
    for b_tn, b_fp, b_fn, b_tp in boot_counts:
        records.append(metrics_from_counts(b_tn, b_fp, b_fn, b_tp))

    boot_df = pd.DataFrame(records)
    point = metrics_from_counts(tn, fp, fn, tp)

    out = {}
    for metric in boot_df.columns:
        vals = boot_df[metric].dropna().to_numpy()
        out[f"{metric}"] = point[metric]
        out[f"{metric}_ci_low"] = float(np.percentile(vals, 2.5))
        out[f"{metric}_ci_high"] = float(np.percentile(vals, 97.5))
    return out

def add_count_bootstrap_cis(df, context, B=2000, seed=42):
    rows = []
    for i, row in df.iterrows():
        ci = bootstrap_ci_from_confusion_counts(
            row["tn"], row["fp"], row["fn"], row["tp"],
            B=B,
            seed=seed + i
        )
        base = {
            "context": context,
            "setting": row.get("setting", ""),
            "model": row.get("model", ""),
            "treatment": row.get("treatment", context),
            "tn": int(row["tn"]),
            "fp": int(row["fp"]),
            "fn": int(row["fn"]),
            "tp": int(row["tp"]),
            "bootstrap_B": B,
            "bootstrap_type": "multinomial_confusion_category_resampling",
        }
        rows.append({**base, **ci})
    return pd.DataFrame(rows)

print("Bootstrap functions ready.")


Bootstrap functions ready.


In [8]:
# وظيفة الخلية: حساب Bootstrap CIs للمقاييس الأساسية من جدول النتائج الرئيسي وجدول attack-only.

ci_main = add_count_bootstrap_cis(
    metrics_main,
    context="directory_protocol",
    B=COUNT_BOOTSTRAP_B,
    seed=RANDOM_SEED
)

ci_attack_only = pd.DataFrame()
if not attack_only_metrics.empty:
    ci_attack_only = add_count_bootstrap_cis(
        attack_only_metrics,
        context="attack_only_excluding_benign",
        B=COUNT_BOOTSTRAP_B,
        seed=RANDOM_SEED + 10_000
    )

ci_all_threshold = pd.concat([ci_main, ci_attack_only], ignore_index=True)

threshold_out_path = os.path.join(OUTPUT_PARENT, "threshold_metric_bootstrap_ci_all_models.csv")
ci_all_threshold.to_csv(threshold_out_path, index=False)

headline = ci_all_threshold[
    (ci_all_threshold["context"] == "directory_protocol")
    & (ci_all_threshold["setting"] == TARGET_SETTING)
    & (ci_all_threshold["model"] == TARGET_MODEL)
].copy()

headline_path = os.path.join(OUTPUT_PARENT, "headline_ocsvm_dedup_on_threshold_ci.csv")
headline.to_csv(headline_path, index=False)

print("Saved:", threshold_out_path)
print("Saved:", headline_path)

display_cols = [
    "context", "setting", "model", "treatment",
    "f1", "f1_ci_low", "f1_ci_high",
    "mcc", "mcc_ci_low", "mcc_ci_high",
    "fpr", "fpr_ci_low", "fpr_ci_high",
    "balanced_accuracy", "balanced_accuracy_ci_low", "balanced_accuracy_ci_high",
]
print("\\nHeadline CI:")
print(headline[display_cols].to_string(index=False))


Saved: /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip/threshold_metric_bootstrap_ci_all_models.csv
Saved: /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip/headline_ocsvm_dedup_on_threshold_ci.csv
\nHeadline CI:
           context  setting model          treatment       f1  f1_ci_low  f1_ci_high      mcc  mcc_ci_low  mcc_ci_high      fpr  fpr_ci_low  fpr_ci_high  balanced_accuracy  balanced_accuracy_ci_low  balanced_accuracy_ci_high
directory_protocol dedup_on ocsvm directory_protocol 0.998887   0.998853    0.998924 0.973721    0.972898     0.974596 0.049756    0.048112     0.051294           0.975122                  0.974353                   0.975944


In [9]:
# وظيفة الخلية: تحميل تنبؤات النموذج الرئيسي إذا أردنا حساب فواصل ثقة للـ ROC-AUC و PR-AUC من anomaly_score.

def find_prediction_file(results_root):
    parquet_path = os.path.join(results_root, "predictions", "test_predictions.parquet")
    csv_path = os.path.join(results_root, "predictions", "test_predictions.csv")
    if os.path.exists(parquet_path):
        return parquet_path
    if os.path.exists(csv_path):
        return csv_path
    return None

def load_target_predictions(pred_path, setting, model):
    usecols = ["__source_file", "final_label", "anomaly_score", "prediction", "model", "setting"]
    if pred_path.endswith(".parquet"):
        print("Reading parquet predictions:", pred_path)
        df = pd.read_parquet(pred_path, columns=usecols)
        df = df[(df["setting"] == setting) & (df["model"] == model)].copy()
        return df

    print("Reading CSV predictions in chunks:", pred_path)
    chunks = []
    for idx, chunk in enumerate(pd.read_csv(pred_path, usecols=usecols, chunksize=CSV_CHUNKSIZE), 1):
        sub = chunk[(chunk["setting"] == setting) & (chunk["model"] == model)].copy()
        if len(sub):
            chunks.append(sub)
        if idx % 5 == 0:
            print("Processed chunks:", idx, "kept rows:", sum(len(c) for c in chunks))
    if not chunks:
        raise ValueError("No predictions matched the requested setting/model")
    return pd.concat(chunks, ignore_index=True)

pred_path = find_prediction_file(RESULTS_ROOT)
print("Prediction file:", pred_path)

target_predictions = None
if RUN_SCORE_BASED_AUC_CI:
    if pred_path is None:
        print("No prediction file found. Score-based AUC CI will be skipped.")
    else:
        target_predictions = load_target_predictions(pred_path, TARGET_SETTING, TARGET_MODEL)
        print("Target predictions:", target_predictions.shape)
        print(target_predictions["final_label"].value_counts().sort_index())


Prediction file: /content/bootstrap_ci_work/extracted_results/predictions/test_predictions.parquet
Reading parquet predictions: /content/bootstrap_ci_work/extracted_results/predictions/test_predictions.parquet
Target predictions: (1684304, 6)
final_label
0      72193
1    1612111
Name: count, dtype: int64


In [10]:
# وظيفة الخلية: حساب Bootstrap CIs للـ ROC-AUC و PR-AUC من anomaly_score للنموذج الرئيسي فقط.

def bootstrap_score_cis(df, label_col="final_label", score_col="anomaly_score", B=200, seed=42):
    rng = np.random.default_rng(seed)

    y = df[label_col].to_numpy(dtype=np.int8)
    s = df[score_col].to_numpy(dtype=float)

    pos_scores = s[y == 1]
    neg_scores = s[y == 0]

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        raise ValueError("Both positive and negative classes are required for ROC-AUC/PR-AUC.")

    point_y = np.concatenate([np.ones(len(pos_scores), dtype=np.int8), np.zeros(len(neg_scores), dtype=np.int8)])
    point_s = np.concatenate([pos_scores, neg_scores])

    point_roc_auc = float(roc_auc_score(point_y, point_s))
    point_pr_auc = float(average_precision_score(point_y, point_s))

    roc_vals = []
    pr_vals = []

    for b in range(B):
        pos_idx = rng.integers(0, len(pos_scores), size=len(pos_scores))
        neg_idx = rng.integers(0, len(neg_scores), size=len(neg_scores))

        bs = np.concatenate([pos_scores[pos_idx], neg_scores[neg_idx]])
        by = np.concatenate([
            np.ones(len(pos_idx), dtype=np.int8),
            np.zeros(len(neg_idx), dtype=np.int8)
        ])

        roc_vals.append(float(roc_auc_score(by, bs)))
        pr_vals.append(float(average_precision_score(by, bs)))

        if (b + 1) % max(1, B // 10) == 0:
            print(f"Score bootstrap progress: {b + 1}/{B}")

    return {
        "roc_auc": point_roc_auc,
        "roc_auc_ci_low": float(np.percentile(roc_vals, 2.5)),
        "roc_auc_ci_high": float(np.percentile(roc_vals, 97.5)),
        "pr_auc": point_pr_auc,
        "pr_auc_ci_low": float(np.percentile(pr_vals, 2.5)),
        "pr_auc_ci_high": float(np.percentile(pr_vals, 97.5)),
        "positive_rows": int(len(pos_scores)),
        "negative_rows": int(len(neg_scores)),
        "bootstrap_B": int(B),
        "bootstrap_type": "stratified_row_resampling_by_class_on_scores",
    }

score_ci_records = []

if RUN_SCORE_BASED_AUC_CI and target_predictions is not None:
    print("Computing score-based CI for directory protocol...")
    score_ci_directory = bootstrap_score_cis(
        target_predictions,
        B=SCORE_BOOTSTRAP_B,
        seed=RANDOM_SEED + 20_000
    )
    score_ci_records.append({
        "context": "directory_protocol",
        "setting": TARGET_SETTING,
        "model": TARGET_MODEL,
        **score_ci_directory
    })

    if "__source_file" in target_predictions.columns:
        attack_only_df = target_predictions[target_predictions["__source_file"] != "Benign_test.pcap.csv"].copy()
        print("\\nComputing score-based CI for attack-only excluding Benign_test.pcap.csv...")
        score_ci_attack_only = bootstrap_score_cis(
            attack_only_df,
            B=SCORE_BOOTSTRAP_B,
            seed=RANDOM_SEED + 30_000
        )
        score_ci_records.append({
            "context": "attack_only_excluding_benign",
            "setting": TARGET_SETTING,
            "model": TARGET_MODEL,
            **score_ci_attack_only
        })

score_ci_df = pd.DataFrame(score_ci_records)

score_out_path = os.path.join(OUTPUT_PARENT, "score_auc_pr_bootstrap_ci_headline.csv")
if not score_ci_df.empty:
    score_ci_df.to_csv(score_out_path, index=False)
    print("\\nSaved:", score_out_path)
    print(score_ci_df.to_string(index=False))
else:
    print("Score-based CI skipped or unavailable.")


Computing score-based CI for directory protocol...
Score bootstrap progress: 20/200
Score bootstrap progress: 40/200
Score bootstrap progress: 60/200
Score bootstrap progress: 80/200
Score bootstrap progress: 100/200
Score bootstrap progress: 120/200
Score bootstrap progress: 140/200
Score bootstrap progress: 160/200
Score bootstrap progress: 180/200
Score bootstrap progress: 200/200
\nComputing score-based CI for attack-only excluding Benign_test.pcap.csv...
Score bootstrap progress: 20/200
Score bootstrap progress: 40/200
Score bootstrap progress: 60/200
Score bootstrap progress: 80/200
Score bootstrap progress: 100/200
Score bootstrap progress: 120/200
Score bootstrap progress: 140/200
Score bootstrap progress: 160/200
Score bootstrap progress: 180/200
Score bootstrap progress: 200/200
\nSaved: /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip/score_auc_pr_bootstrap_ci_headline.csv
                   

In [11]:
# وظيفة الخلية: إنشاء جدول مختصر جاهز للاستخدام داخل الورقة من فواصل الثقة المحسوبة.

summary_rows = []

# صف المقاييس threshold-based للنتيجة الرئيسية.
if not headline.empty:
    h = headline.iloc[0].to_dict()
    summary_rows.append({
        "context": h["context"],
        "setting": h["setting"],
        "model": h["model"],
        "metric": "F1",
        "point": h["f1"],
        "ci_low": h["f1_ci_low"],
        "ci_high": h["f1_ci_high"],
        "bootstrap_B": h["bootstrap_B"],
    })
    summary_rows.append({
        "context": h["context"],
        "setting": h["setting"],
        "model": h["model"],
        "metric": "MCC",
        "point": h["mcc"],
        "ci_low": h["mcc_ci_low"],
        "ci_high": h["mcc_ci_high"],
        "bootstrap_B": h["bootstrap_B"],
    })
    summary_rows.append({
        "context": h["context"],
        "setting": h["setting"],
        "model": h["model"],
        "metric": "FPR",
        "point": h["fpr"],
        "ci_low": h["fpr_ci_low"],
        "ci_high": h["fpr_ci_high"],
        "bootstrap_B": h["bootstrap_B"],
    })
    summary_rows.append({
        "context": h["context"],
        "setting": h["setting"],
        "model": h["model"],
        "metric": "Balanced accuracy",
        "point": h["balanced_accuracy"],
        "ci_low": h["balanced_accuracy_ci_low"],
        "ci_high": h["balanced_accuracy_ci_high"],
        "bootstrap_B": h["bootstrap_B"],
    })

# صفوف AUC/AP إذا حُسبت.
if not score_ci_df.empty:
    for _, r in score_ci_df.iterrows():
        summary_rows.append({
            "context": r["context"],
            "setting": r["setting"],
            "model": r["model"],
            "metric": "ROC-AUC",
            "point": r["roc_auc"],
            "ci_low": r["roc_auc_ci_low"],
            "ci_high": r["roc_auc_ci_high"],
            "bootstrap_B": r["bootstrap_B"],
        })
        summary_rows.append({
            "context": r["context"],
            "setting": r["setting"],
            "model": r["model"],
            "metric": "PR-AUC",
            "point": r["pr_auc"],
            "ci_low": r["pr_auc_ci_low"],
            "ci_high": r["pr_auc_ci_high"],
            "bootstrap_B": r["bootstrap_B"],
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_PARENT, "bootstrap_ci_summary_for_manuscript.csv")
summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
print(summary_df.to_string(index=False))


Saved: /content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip/bootstrap_ci_summary_for_manuscript.csv
                     context  setting model            metric    point   ci_low  ci_high  bootstrap_B
          directory_protocol dedup_on ocsvm                F1 0.998887 0.998853 0.998924         2000
          directory_protocol dedup_on ocsvm               MCC 0.973721 0.972898 0.974596         2000
          directory_protocol dedup_on ocsvm               FPR 0.049756 0.048112 0.051294         2000
          directory_protocol dedup_on ocsvm Balanced accuracy 0.975122 0.974353 0.975944         2000
          directory_protocol dedup_on ocsvm           ROC-AUC 1.000000 1.000000 1.000000          200
          directory_protocol dedup_on ocsvm            PR-AUC 1.000000 1.000000 1.000000          200
attack_only_excluding_benign dedup_on ocsvm           ROC-AUC 1.000000 1.000000 1.000000          200
atta

In [12]:
# وظيفة الخلية: حفظ إعدادات التشغيل وضغط نتائج Bootstrap في ملف ZIP لرفعه أو إرساله للمراجعة.

config = {
    "source_info": SOURCE_INFO,
    "results_root_used": RESULTS_ROOT,
    "output_parent": OUTPUT_PARENT,
    "count_bootstrap_B": COUNT_BOOTSTRAP_B,
    "run_score_based_auc_ci": RUN_SCORE_BASED_AUC_CI,
    "score_bootstrap_B": SCORE_BOOTSTRAP_B,
    "target_setting": TARGET_SETTING,
    "target_model": TARGET_MODEL,
    "random_seed": RANDOM_SEED,
    "bootstrap_ci_note": (
        "Threshold metrics use multinomial resampling over observed confusion-matrix categories. "
        "Score metrics use stratified row resampling by class over anomaly_score."
    ),
}

config_path = os.path.join(OUTPUT_PARENT, "bootstrap_ci_run_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

timestamp = time.strftime("%Y%m%d_%H%M%S")
zip_path = os.path.join(OUTPUT_PARENT, f"bootstrap_ci_results_{timestamp}.zip")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUT_PARENT):
        for f in files:
            if f.endswith(".zip"):
                continue
            fp = os.path.join(root, f)
            arc = os.path.relpath(fp, OUTPUT_PARENT)
            z.write(fp, arc)

print("Bootstrap CI outputs saved in:")
print(OUTPUT_PARENT)
print("\\nZIP package:")
print(zip_path)

print("\\nFiles:")
for f in sorted(os.listdir(OUTPUT_PARENT)):
    print("-", f)


Bootstrap CI outputs saved in:
/content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip
\nZIP package:
/content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/bootstrap_ci_results_from_zip/bootstrap_ci_results_20260507_000824.zip
\nFiles:
- bootstrap_ci_results_20260507_000824.zip
- bootstrap_ci_run_config.json
- bootstrap_ci_summary_for_manuscript.csv
- headline_ocsvm_dedup_on_threshold_ci.csv
- score_auc_pr_bootstrap_ci_headline.csv
- threshold_metric_bootstrap_ci_all_models.csv


## الملفات التي ترسلها بعد التشغيل

بعد انتهاء التشغيل، أرسل الملفات التالية:

1. `bootstrap_ci_summary_for_manuscript.csv`
2. `threshold_metric_bootstrap_ci_all_models.csv`
3. `headline_ocsvm_dedup_on_threshold_ci.csv`
4. `score_auc_pr_bootstrap_ci_headline.csv` إذا تم حسابه
5. `bootstrap_ci_run_config.json`
6. ملف ZIP الناتج `bootstrap_ci_results_*.zip`

إذا فشل البحث التلقائي عن النتائج في Drive، عدّل أحد المتغيرين في خلية الإعدادات:

```python
MANUAL_RESULTS_ROOT = "/content/drive/MyDrive/.../publication_full_..."
```

أو:

```python
MANUAL_RESULTS_ZIP = "/content/drive/MyDrive/.../publication_full_20260506_013718.zip"
```
